# 07 — Expert system

The rule base is a list of IF/THEN checks. Forward chaining fires each matching rule once and keeps the rule ids as the explanation. Before takeoff, R1 stops the mission in a storm, and R2/R3 reject a target that sits in a restricted cell or that DFS cannot reach from the base.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
from uavplanner.config import Config
from uavplanner.expert import default_rules, forward_chain, static_restricted
from uavplanner.uninformed import dfs_reachable
from uavplanner.world import make_targets, make_world

rules = default_rules()
for speed in (6.0, 14.0):
    facts, fired = forward_chain({'wind_speed': speed}, rules)
    print(f'{speed:.0f} m/s -> fired {fired or ["none"]}, verdict {facts.get("verdict", "GO")}')

cfg = Config(demo_rejections=True, grid_n=30, seed=4)
rng = np.random.default_rng(cfg.seed)
world = make_world(cfg, rng)
targets = make_targets(world, cfg, rng)
blocked = static_restricted(world, cfg)
reachable = dfs_reachable(blocked, world.base)
print(f'restricted cells: {int(blocked.sum())}')
for target in targets:
    facts, fired = forward_chain(
        {
            'target_restricted': bool(blocked[target.pos]),
            'target_reachable': bool(reachable[target.pos]),
        },
        rules,
    )
    if fired:
        print(f'target {target.id} at {target.pos}: {fired} {facts.get("reasons")}')
